# TrollLens: Veri Hazırlığı ve Özellik Mühendisliği

Bu notebook, TrollLens projesinin (Marka İtibarına Yönelik Koordineli Sahte Etkileşimlerin Davranışsal Tespiti) makine öğrenmesi süreçlerinin 1. Aşamasını (Veri Hazırlığı ve Özellik Mühendisliği) ve 2. Aşamasının başını (Model Seçimi) içermektedir.

## 1. Overview (Genel Bakış)

Veri hazırlığı ve özellik mühendisliği, modelimizin koordineli bot ağlarını (trolleri) organik kullanıcılardan ayırabilmesi için en kritik aşamadır. Marka itibarına yönelik saldırılar (astroturfing) genellikle içerik olarak organik müşteri şikayetlerine benzese de, **davranışsal zaman serilerinde** (aynı anda tweet atma, yüksek retweet oranları) belirgin imzalar bırakırlar. Bu aşamada, ham veriyi modelin bu davranışsal imzaları öğrenebileceği formata dönüştürüyoruz.

## 2. Data Collection (Veri Toplama)

Projede, kurumsal saldırıların davranışsal proxy'si olarak akademik Information Operations veri seti kullanılmaktadır. Orijinal veri seti çok büyük (~8 GB) olduğu için, bellek sınırlarını aşmamak ve eğitim süresini optimize etmek adına Python'da `pandas chunksize` ve **Tabakalı (Stratified) Örnekleme** kullanılarak hedef sınıf (`info_op`) dağılımı korunacak şekilde %25 oranında küçültülmüş ve `troll_lens_subset.csv` dosyası elde edilmiştir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os

# Görselleştirme ayarları
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Veri setini yükleme
data_path = os.path.join('archive', 'troll_lens_subset.csv')
df = pd.read_csv(data_path, low_memory=False)
print(f"Veri seti boyutu: {df.shape}")
display(df.head())

## 3. Data Cleaning (Veri Temizliği)

Verideki eksik değerleri kontrol edip gidereceğiz. Ayrıca, projemiz **dilden bağımsız davranışsal analize** odaklandığı için kelime çantası (`BoW` - Bag of Words) gibi NLP (Doğal Dil İşleme) sütunlarını modelden dışlayacağız. Zaman verilerini modele uygun tarih formatına dönüştüreceğiz.

In [ ]:
# Eksik değer kontrolü
print("Eksik Değerler:\n", df.isnull().sum())

# Dilden bağımsız model için BoW sütununu düşürüyoruz
if 'BoW' in df.columns:
    df.drop(columns=['BoW'], inplace=True)

# NaN değerleri sayısal sütunlar için 0 ile dolduralım (Eğer RT/Like sayısı yoksa 0'dır)
num_cols = ['quote_count', 'like_count', 'retweet_count', 'hashtags', 'urls', 'user_mentions']
df[num_cols] = df[num_cols].fillna(0)

# Tarih sütununu datetime formatına çeviriyoruz
df['tweet_time'] = pd.to_datetime(df['tweet_time'], errors='coerce')

# Tarihi eksik olan satırları (varsa) düşürüyoruz (zaman serisi analizi için kritik)
df = df.dropna(subset=['tweet_time'])

print("Temizlenmiş veri boyutu:", df.shape)

## 4. Exploratory Data Analysis (Keşifsel Veri Analizi - EDA)

Verinin dağılımını, organik vs. troll etkileşim davranışlarını ve koordineli saldırıların zaman serisindeki izlerini analiz ediyoruz.

In [ ]:
# 1. Sınıf Dağılımı (Troll vs Organik)
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='info_op', palette='viridis')
plt.title('Hesap Türü Dağılımı (0: Organik, 1: Troll/Info Op)')
plt.show()

# 2. Etkileşim Metriklerinin Karşılaştırılması (Retweet & Like)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.barplot(data=df, x='info_op', y='retweet_count', palette='viridis', ci=None)
plt.title('Hesap Türüne Göre Ortalama Retweet')

plt.subplot(1, 2, 2)
sns.barplot(data=df, x='info_op', y='like_count', palette='viridis', ci=None)
plt.title('Hesap Türüne Göre Ortalama Beğeni (Like)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Davranışsal Zaman Serisi: Trollerin ve Organik hesapların aktivite saatleri
df['hour'] = df['tweet_time'].dt.hour

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='hour', hue='info_op', bins=24, multiple="dodge", palette='viridis')
plt.title('Günün Saatlerine Göre Tweet Atma Frekansı (Troll vs Organik)')
plt.xlabel('Günün Saati (0-23)')
plt.ylabel('Tweet Sayısı')
plt.show()

## 5. Feature Engineering (Özellik Mühendisliği)

Mevcut özelliklerden yola çıkarak modelin saldırı davranışını yakalamasını sağlayacak yeni değişkenler (feature) üretiyoruz.
1. **`total_engagement`**: Bir tweetin aldığı toplam tepki (retweet + like + quote).
2. **`is_weekend`**: Hafta sonu atılan tweetlerin tespiti (troller bazen mesai dışı / hafta sonu davranış kalıpları gösterir).
3. **`time_of_day`**: Günün bölümleri (Sabah, Öğle, Akşam, Gece). Trollerin koordineli saldırıları genellikle günün belli zaman dilimlerinde yoğunlaşır.

In [ ]:
# Toplam etkileşim (Engagement)
df['total_engagement'] = df['like_count'] + df['retweet_count'] + df['quote_count']

# Zaman bazlı davranış özellikleri
df['day_of_week'] = df['tweet_time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return 1 # Sabah
    elif 12 <= hour < 17:
        return 2 # Öğle
    elif 17 <= hour < 22:
        return 3 # Akşam
    else:
        return 4 # Gece

df['time_of_day'] = df['hour'].apply(get_time_of_day)

display(df[['tweet_time', 'total_engagement', 'is_weekend', 'time_of_day', 'info_op']].head())

## 6. Data Transformation (Veri Dönüştürme)

Makine öğrenmesi algoritmaları, özelliklerin aynı ölçekte olmasına ihtiyaç duyar. Ayrıca `tweetid`, `userid` gibi hashlenmiş id değerlerini (GNN kullanmadığımız senaryoda) modele vermemek gerekir, aksi halde ezber (overfitting) yapar. 
Burada sadece davranışsal sayısal değişkenleri ölçeklendiriyoruz (Standardization).

In [ ]:
features_to_scale = ['quote_count', 'like_count', 'retweet_count', 'hashtags', 'urls', 
                     'user_mentions', 'total_engagement', 'hour']

# StandardScaler ile ortalamayı 0, standart sapmayı 1 yapacak şekilde ölçeklendirme
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Ölçeklendirme sonrası özelliklerin ilk 5 satırı:")
display(df_scaled[features_to_scale].head())

## Model Exploration - 1. Model Selection (Model Seçimi)

### Seçilen Model: GNN (Graph Neural Networks) + Zaman Serisi Analizi ve Ağaç Tabanlı Temel (Baseline) Model

TrollLens projesi için marka krizlerini (koordineli saldırıları) tespit etmede geleneksel makine öğrenmesi modelleri yetersiz kalabilir, çünkü saldırıların en önemli özelliği "tekil tweetlerin içeriği" değil, "kullanıcıların birbiriyle kurduğu toplu etkileşim ağıdır" (örneğin; birbirini anında retweetleyen yüzlerce bot hesap).

**1. Baseline Model (XGBoost veya Random Forest):** 
* **Neden?** GNN gibi kompleks bir modele geçmeden önce referans noktası (baseline) oluşturmak için. Özellik mühendisliğiyle yarattığımız zaman (hour) ve metrik (total_engagement) özellikleri üzerinden hesapları sınıflandırır.
* **Artıları:** Hızlı eğitilir, feature importance (hangi özelliğin daha önemli olduğu) rahatlıkla görülür.
* **Eksileri:** Hesaplar arasındaki ağ ve ilişki (network) topolojisini göremez.

**2. Ana Model (Graph Neural Networks - GNN):**
* **Neden?** Troll hesapların çalışma prensibi GNN mantığıyla birebir uyuşur. Düğümler (Nodes) kullanıcıları, kenarlar (Edges) ise aralarındaki retweet/mention etkileşimlerini temsil eder.
* **Artıları:** Bir hesap organik görünse bile, sürekli olarak koordineli bir troll ağıyla etkileşime giriyorsa GNN bunun davranışsal imzasını (astroturfing) yakalayabilir. Projemizin temel tezi olan "Ne söylediklerinden ziyade, nasıl hareket ettikleri" prensibini tam olarak karşılar.
* **Eksileri:** Veri hazırlığı ve eğitimi (özellikle büyük matris işlemleri) çok daha yüksek hesaplama gücü (RAM/GPU) gerektirir.